In [1]:
from airflow.sdk import dag, task
from datetime import datetime, timedelta
from airflow import DAG
from airflow.providers.standard.operators.python import PythonOperator
from airflow.providers.standard.operators.empty import EmptyOperator
from airflow.providers.standard.operators.bash import BashOperator
from airflow.sdk import Param
from airflow.sdk.definitions.param import ParamsDict
from sqlalchemy import create_engine, VARCHAR, Integer,Date, inspect
import os
import pandas as pd
from datetime import datetime
import zipfile
import numpy as np
import requests
from datetime import datetime
from pathlib import Path
import shutil
import boto3
from botocore.config import Config


c:\Users\Nelio\anaconda3\Lib\site-packages\airflow\__init__.py:45: RuntimeWarning: Airflow currently can be run on POSIX-compliant Operating Systems. For development, it is regularly tested on fairly modern Linux Distros and recent versions of macOS. On Windows you can run it via WSL2 (Windows Subsystem for Linux 2) or via Linux Containers. The work to add Windows support is tracked via https://github.com/apache/airflow/issues/10388, but it is not a high priority.
  warnings.warn(
2026-04-15T14:39:48.336308Z [info     ] setup plugin alembic.autogenerate.schemas [alembic.runtime.plugins] loc=plugins.py:37
2026-04-15T14:39:48.344273Z [info     ] setup plugin alembic.autogenerate.tables [alembic.runtime.plugins] loc=plugins.py:37
2026-04-15T14:39:48.351591Z [info     ] setup plugin alembic.autogenerate.types [alembic.runtime.plugins] loc=plugins.py:37
2026-04-15T14:39:48.359553Z [info     ] setup plugin alembic.autogenerate.constraints [alembic.runtime.plugins] loc=plugins.py:37
2026-04-1

In [8]:
def candidato(file_csv):

    dfs = []
    lista_links = []
    # Configurações do Cloudflare
    account_id = '8e2ebba9301e868cfdedc13086d6c140'
    access_key = 'a35b4c037c701ade40b95d0c48e4b4b3'
    secret_key = '9eb9f6eac22bb9a13a90f36ab18092e455fef1ead93ec303fe94c0bf2ae90e42'
    bucket_name = 'tre'
    folder_local = r'C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\candidato' ##os.getenv('CANDIDATO_FOTOS_DIR', '/opt/airflow/TRE2026/archives/RR/candidato/jpg')
    folder_remote = 'fotosrr/'               

    s3 = boto3.client(
    service_name='s3',
    endpoint_url=f'https://{account_id}.r2.cloudflarestorage.com',
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    region_name='auto', 
    config=Config(signature_version='s3v4')
    )
    base_url = "https://pub-e9ede5cdf96443d58340a9a3b62a2816.r2.dev/fotosrr/" 
    # engine = create_engine(f"postgresql+psycopg2://{USER}:{PASSWORD}@db:5432/eleicao_RR")

    for i in os.listdir(file_csv):
        full_path = os.path.join(file_csv, i)
        if full_path.endswith('RR.csv'):
            print(f"Lendo arquivo: {full_path}")
            df = pd.read_csv(full_path, encoding='latin-1', sep=';',dtype=str)
            df['SG_UE'] = df['SG_UE'].astype(str).str.lstrip("0")
            df.loc[df['NM_UE'] == "RORAIMA", 'SG_UE'] = '3018'
            df['NM_CANDIDATO'] = (
                df['NM_URNA_CANDIDATO']
                .fillna('')
                .astype(str)
                .str.normalize('NFKD')
                .str.encode('ascii', 'ignore')
                .str.decode('utf-8')
                .str.replace(r'[^A-Za-z0-9]+', '', regex=True)
            )
            df['ID_CANDIDATO'] = df['ANO_ELEICAO'] + df['NR_TURNO'] +  df['CD_CARGO'] + df['SG_UE'] +  df['NR_CANDIDATO'] + df['NM_CANDIDATO']
            dfs.append(df)


    dfconsulta = pd.concat(dfs, ignore_index=True)

    # for file in os.listdir(folder_local):
    #     if file.lower().endswith(('.png', '.jpg', '.jpeg')):
    #         lista_links.append({
    #             "Nome_Foto": file,
    #             "URL_Foto": base_url + file
    #         })
    # fotos = pd.DataFrame(lista_links)
    # fotos['idCandidatos'] = fotos['Nome_Foto'].str[2:14]

    # df = pd.merge(
    # dfconsulta, 
    # fotos, 
    # how='left', 
    # right_on='idCandidatos',
    # left_on='SQ_CANDIDATO').drop('idCandidatos', axis=1)
    # df.columns = [col.lower() for col in df.columns]
    # df.drop_duplicates(['id_candidato'], inplace=True)

    # df.to_sql("dcandidato", engine, index=False, if_exists='append')





    return dfconsulta

In [9]:
df = candidato(file_csv = r'C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\candidato\csv')


Lendo arquivo: C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\candidato\csv\consulta_cand_2020_RR.csv
Lendo arquivo: C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\candidato\csv\consulta_cand_2022_RR.csv
Lendo arquivo: C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\candidato\csv\consulta_cand_2024_RR.csv


In [10]:
df

,DT_GERACAO,HH_GERACAO,ANO_ELEICAO,CD_TIPO_ELEICAO,NM_TIPO_ELEICAO,NR_TURNO,CD_ELEICAO,DS_ELEICAO,DT_ELEICAO,TP_ABRANGENCIA,...,DS_GRAU_INSTRUCAO,CD_ESTADO_CIVIL,DS_ESTADO_CIVIL,CD_COR_RACA,DS_COR_RACA,CD_OCUPACAO,DS_OCUPACAO,CD_SIT_TOT_TURNO,DS_SIT_TOT_TURNO,ID_CANDIDATO
0,06/01/2026,16:44:45,2020,2,ELEIÇÃO ORDINÁRIA,1,426,Eleições Municipais 2020,15/11/2020,MUNICIPAL,...,ENSINO MÉDIO COMPLETO,9,DIVORCIADO(A),03,PARDA,999,OUTROS,5,SUPLENTE,2020113302622333DANIELEDECAMPOSNOVOS
1,06/01/2026,16:44:45,2020,2,ELEIÇÃO ORDINÁRIA,1,426,Eleições Municipais 2020,15/11/2020,MUNICIPAL,...,ENSINO MÉDIO COMPLETO,1,SOLTEIRO(A),03,PARDA,999,OUTROS,5,SUPLENTE,2020113302622222CASSIODOTAXI
2,06/01/2026,16:44:45,2020,2,ELEIÇÃO ORDINÁRIA,1,426,Eleições Municipais 2020,15/11/2020,MUNICIPAL,...,ENSINO FUNDAMENTAL INCOMPLETO,1,SOLTEIRO(A),05,INDÍGENA,601,AGRICULTOR,5,SUPLENTE,2020113302611611MARA
3,06/01/2026,16:44:45,2020,2,ELEIÇÃO ORDINÁRIA,1,426,Eleições Municipais 2020,15/11/2020,MUNICIPAL,...,SUPERIOR COMPLETO,1,SOLTEIRO(A),03,PARDA,999,OUTROS,5,SUPLENTE,2020113302610555TANDYLIMA
4,06/01/2026,16:44:45,2020,2,ELEIÇÃO ORDINÁRIA,1,426,Eleições Municipais 2020,15/11/2020,MUNICIPAL,...,SUPERIOR COMPLETO,3,CASADO(A),03,PARDA,266,PROFESSOR DE ENSINO MÉDIO,5,SUPLENTE,2020113302610456PROFESSORAGNALDO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3845,24/03/2026,19:31:03,2024,2,ELEIÇÃO ORDINÁRIA,1,619,Eleições Municipais 2024,06/10/2024,MUNICIPAL,...,ENSINO MÉDIO COMPLETO,1,SOLTEIRO(A),01,BRANCA,999,OUTROS,5,SUPLENTE,2024113305010000RONYPETERSON
3846,24/03/2026,19:31:03,2024,2,ELEIÇÃO ORDINÁRIA,1,619,Eleições Municipais 2024,06/10/2024,MUNICIPAL,...,SUPERIOR COMPLETO,1,SOLTEIRO(A),03,PARDA,999,OUTROS,2,ELEITO POR QP,2024113305010777SANDRIELYCUNHA
3847,24/03/2026,19:31:03,2024,2,ELEIÇÃO ORDINÁRIA,1,619,Eleições Municipais 2024,06/10/2024,MUNICIPAL,...,SUPERIOR COMPLETO,1,SOLTEIRO(A),01,BRANCA,931,"ESTUDANTE, BOLSISTA, ESTAGIÁRIO E ASSEMELHADOS",5,SUPLENTE,2024113305020789MOACYPEDEMINGAL
3848,24/03/2026,19:31:03,2024,2,ELEIÇÃO ORDINÁRIA,1,619,Eleições Municipais 2024,06/10/2024,MUNICIPAL,...,SUPERIOR COMPLETO,3,CASADO(A),03,PARDA,257,EMPRESÁRIO,5,SUPLENTE,2024113305020333CHEILADOAUAU
